# Day 2: API Cleaning - 05_axis (119092)
**Instructions:**
- Parse dates
- Forward-fill missing NAV
- Validate NAV > 0

In [2]:
import pandas as pd
import os

amfi_code = 119092
raw_path = f'../../data/raw/nav_{amfi_code}.csv'
processed_dir = '../../data/processed/'

df = pd.read_csv(raw_path)
print(f"Initial rows: {len(df)}")
df.head()

Initial rows: 3565


,date,nav
0,01-06-2026,6156.7532
1,29-05-2026,6151.1139
2,27-05-2026,6146.6118
3,26-05-2026,6144.0004
4,25-05-2026,6144.8478


In [3]:
df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
df = df.drop_duplicates(subset=['date'])
df = df[df['nav'] > 0]
df = df.sort_values('date')

# Forward fill for weekends/holidays
min_date = df['date'].min()
max_date = df['date'].max()
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')
df = df.set_index('date').reindex(all_dates)
df['nav'] = df['nav'].ffill()
df['amfi_code'] = amfi_code
df.index.name = 'date'
df = df.reset_index()

print(f"Final rows: {len(df)}")

ValueError: time data "29-05-2026" doesn't match format "%m-%d-%Y", at position 1. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
output_file = f'api_{amfi_code}_cleaned.csv'
df.to_csv(os.path.join(processed_dir, output_file), index=False)
print(f"Saved to {output_file}")